# Chapitre 7 - Lab : niveaux d'isolation réels de PostgreSQL

Les cours 2 à 4 ont rejoué les anomalies sur un mini-moteur. Ce lab les reproduit sur un
vrai système, PostgreSQL 16, avec deux connexions (ou plus) qui s'entrelacent : lectures
sales impossibles, lecture biaisée en `READ COMMITTED`, versions MVCC visibles
(`xmin`, `xmax`, `ctid`, pages brutes), mises à jour perdues et leurs remèdes, write skew
des médecins de garde, fantôme de la réservation de salle, échecs de sérialisation
(`SQLSTATE 40001`) et interblocages (`40P01`).

**Démarrer et arrêter PostgreSQL** (depuis la racine du dépôt) :

```bash
docker compose -f docker/transactions/compose.yml -p ddia_transactions up -d
# ... exécuter le notebook ...
docker compose -f docker/transactions/compose.yml -p ddia_transactions down -v
```

Le conteneur écoute sur le port **10832** de l'hôte (utilisateur, mot de passe et base :
`ddia`).

**Plan.**

1. Pas de lecture sale, même en `READ UNCOMMITTED`.
2. Lecture biaisée en `READ COMMITTED`, instantané en `REPEATABLE READ`.
3. MVCC sous le capot : `xmin`, `xmax`, pages brutes et `VACUUM`.
4. Mises à jour perdues : le problème et quatre remèdes, sous charge.
5. Write skew : les médecins de garde.
6. Fantôme : la réservation de salle.
7. Interblocage détecté.

In [1]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from psycopg import errors

from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
DSN = "host=localhost port=10832 user=ddia password=ddia dbname=ddia connect_timeout=5"
NIVEAUX = {"RC": "READ COMMITTED", "RR": "REPEATABLE READ", "SER": "SERIALIZABLE", "RU": "READ UNCOMMITTED"}

with psycopg.connect(DSN, autocommit=True) as c:
    print(c.execute("SELECT version()").fetchone()[0])


def session(niveau="RC"):
    """Nouvelle connexion dont chaque transaction démarre au niveau demandé."""
    c = psycopg.connect(DSN)
    c.execute(f"SET SESSION CHARACTERISTICS AS TRANSACTION ISOLATION LEVEL {NIVEAUX[niveau]}")
    c.commit()
    return c


def admin(sql, params=None):
    with psycopg.connect(DSN, autocommit=True) as c:
        cur = c.execute(sql, params)
        return cur.fetchall() if cur.description else None


def en_fond(fn, *args):
    """Exécute une instruction qui risque de bloquer dans un fil séparé (renvoie un Future)."""
    ex = ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn, *args)
    ex.shutdown(wait=False)
    return fut

PostgreSQL 16.15 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit
